<a href="https://colab.research.google.com/github/sethkipsangmutuba/Database-Management-System/blob/main/Week_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3: Query Processing & Optimization (I)

## 1. Introduction
In any database management system (DBMS), one of the most important responsibilities is efficiently processing queries. Whether the system is a centralized relational DBMS, a distributed cloud-native system, or a hybrid, its effectiveness depends on how quickly and efficiently it can respond to a user’s query. When a user writes a SQL query, they describe what data they want, not how to retrieve it. The DBMS must decide how to execute the query in the most optimal way possible. This transformation from a high-level declarative query into an efficient low-level execution strategy is known as query processing and optimization. Week 3 focuses on the logical part of that transformation—how SQL queries are parsed, represented as relational algebra, and rewritten using algebraic equivalence and heuristics. The physical aspects, such as cost estimation and operator implementation, will be covered in Week 4.

## 2. Learning Objectives
By the end of this week, you should be able to:
- Explain the stages of query processing from SQL text to an internal execution plan.
- Translate SQL queries into relational algebra expressions.
- Construct logical query trees and interpret their meaning.
- Apply algebraic equivalence rules to transform queries without changing their semantics.
- Use join-order heuristics to propose more efficient query structures.
- Understand the Volcano optimizer model and iterator execution framework.

## 3. Overview of Query Processing
Query processing can be seen as a multi-phase pipeline:
1. Parsing – Check syntax and semantics of SQL.
2. Translation – Convert to a relational algebra expression.
3. Logical Plan Generation – Represent operations in a query tree.
4. Logical Optimization – Apply equivalence rules and heuristics to improve the logical plan.
5. Physical Plan Generation – Choose algorithms and data access paths.
6. Cost Estimation – Predict execution time/resource usage.
7. Plan Selection – Select the cheapest valid plan.  
Week 3 deals with steps 1–4, i.e., the logical side of query optimization.

## 4. Parsing and Syntax Analysis
### 4.1 What is Parsing?
Parsing takes an SQL query and checks:
- Syntax validity – Does the query follow SQL grammar rules?
- Semantic validity – Do the tables, columns, and data types exist and match?  
If valid, the parser produces a parse tree (also called an Abstract Syntax Tree, AST), which structurally represents the query.  
Example:  
`SELECT name, age FROM Students WHERE age > 20 AND major = 'CS';`  
The parse tree represents:
- SELECT clause: name, age
- FROM clause: Students
- WHERE clause: age > 20 AND major = 'CS'

### 4.2 Semantic Checks
- Table/column existence check: Verify if Students exists and contains name, age, major.
- Type checking: Ensure age > 20 compares compatible types.
- Privileges: Verify user has access rights.  
Once parsed and validated, the query moves to translation.

## 5. Translation to Relational Algebra
Relational algebra is a procedural query language used internally by DBMSs. It expresses queries as compositions of algebraic operators.

### 5.1 Core Operators
$\sigma$ (Selection): Filters rows.  
$\pi$ (Projection): Selects columns.  
$\times$ (Cartesian product): All row combinations.  
$\bowtie$ (Join): Combines rows with matching conditions.  
$\cup$ (Union), $\cap$ (Intersection), $-$ (Difference): Set operations.  
$\rho$ (Rename): Renames relations or attributes.

### 5.2 Example Translation
SQL:  
`SELECT name FROM Students WHERE age > 20 AND major = 'CS';`  
Relational Algebra:  
$$
\pi_{\text{name}} \left( \sigma_{\text{age}>20 \wedge \text{major}='CS'} (\text{Students}) \right)
$$  
Selection happens first (filter rows).  
Projection happens last (select final columns).

## 6. Logical Query Trees
A query tree is a tree representation of a relational algebra expression:
- Leaves = base tables.
- Internal nodes = relational operators.  

Example Tree:  
π_name  
&nbsp;&nbsp;|  
σ_age>20 ∧ major='CS'  
&nbsp;&nbsp;|  
Students  

### 6.2 Directed Acyclic Graph (DAG) Representation
For queries with common subexpressions, the DBMS may use a DAG instead of a tree to avoid recomputation.

## 7. Query Plan Representation
Two main types:
- Logical plan: Abstract sequence of algebraic operations (no implementation detail).
- Physical plan: Specifies actual algorithms and access paths (covered next week).

## 8. Algebraic Equivalence Rules
Logical optimization relies on the fact that multiple algebraic expressions can be equivalent in meaning.

### 8.1 Common Rules
Commutativity of joins:  
$$
A \bowtie B \equiv B \bowtie A
$$  
Associativity of joins:  
$$
(A \bowtie B) \bowtie C \equiv A \bowtie (B \bowtie C)
$$  
Cascade of selections:  
$$
\sigma_{a \wedge b}(R) \equiv \sigma_a(\sigma_b(R))
$$  
Selection-Projection exchange:  
$$
\pi_{\text{cols}}(\sigma_{\text{condition}}(R)) \equiv \sigma_{\text{condition}}(\pi_{\text{cols}}(R))
$$  
(if cols include those used in condition)  
Projection pushdown: Remove unused attributes early.

### 8.2 Why These Rules Matter
Pushing selection early → fewer rows in later operations.  
Pushing projection early → fewer columns processed.  
Reordering joins → drastically different intermediate result sizes.

## 9. Join-Order Heuristics
Joins dominate query cost, so join order matters.

### 9.1 Left-Deep Trees
Left-deep join trees are preferred because:
- They allow pipelining.
- The inner relation of each join can be indexed.  

Example left-deep:  
$$
(((A \bowtie B) \bowtie C) \bowtie D)
$$

### 9.2 Right-Deep and Bushy Trees
Right-deep: Useful in parallelism scenarios.  
Bushy: Joins between intermediate results before joining to others—less common.

### 9.3 Heuristic Rules for Joins
Perform most selective joins first (reduce intermediate size quickly).  
Join smaller tables first when possible.  
Use available indexes to avoid full scans.

## 10. Query Rewriting
### 10.1 View Expansion
Queries on views are rewritten by inlining view definitions to allow global optimization.

### 10.2 Redundant Operations
If a DISTINCT is unnecessary due to key constraints, remove it.

### 10.3 Predicate Simplification
Combine or remove redundant WHERE conditions.

## 11. The Volcano Optimizer Framework
### 11.1 Core Concepts
Expressions: Logical or physical.  
Rules: Transform one expression into another.  
Memoization: Avoid re-exploring the same subexpression.

### 11.2 Rule Types
Logical rules: Equivalent transformations (pushdown, join reorder).  
Implementation rules: Replace logical operators with specific algorithms.

## 12. Iterator Model of Execution
In the iterator model (also called pull-based model):
- Every operator supports open(), next(), and close().
- The top operator calls next() on its child, which calls next() on its child, and so on.
- Tuples are processed on demand.

### 12.1 Benefits
Memory-efficient: no need to store all intermediate results.  
Supports pipelining: tuples flow from one operator to the next.

## 13. Step-by-Step Example of Logical Optimization
Suppose:  
`SELECT E.name, D.name FROM Employees E, Departments D WHERE E.dept_id = D.id AND E.salary > 50000 AND D.location = 'Nairobi';`  

Initial logical plan:  
$$
\pi_{E.\text{name}, D.\text{name}} \left( \sigma_{E.\text{salary}>50000 \wedge D.\text{location}='Nairobi'} (E \times D) \right)
$$  

Optimized plan:  
Push selections down:  
$$
\pi_{E.\text{name}, D.\text{name}} \left( \sigma_{E.\text{salary}>50000}(E) \bowtie \sigma_{D.\text{location}='Nairobi'}(D) \right)
$$  
Replace Cartesian + selection with join.  
Push projection down so each branch only carries needed attributes.

## 14. Key Takeaways
Logical optimization transforms queries into more efficient but semantically equivalent forms.  
Query trees and algebraic equivalence rules are central to this process.  
Join-order selection is one of the most important factors for performance.  
The Volcano framework and iterator model underpin many modern optimizers.

## 15. Recommended Reading
Elmasri & Navathe, Fundamentals of Database Systems, Chapters 18–19.  
Selinger et al., Access Path Selection in a Relational DBMS (IBM System R).  
Graefe, Volcano: An Extensible and Parallel Query Evaluation System.


In [ ]:
# ========================================================
# Week 3: Query Processing & Optimization (I)
# Project: Relational Algebra Interpreter & Optimizer
# Features:
#  - SQL parsing
#  - Query tree generation
#  - Algebraic equivalence rewrites
#  - Join order heuristics
#  - Volcano iterator execution model
# Each line includes explanations for students.
# ========================================================

import re                       # For parsing SQL text using regex
from itertools import permutations  # For generating join orders

# ========================================================
# 1. Relational Algebra Tree Node
# ========================================================
class Node:
    """
    Represents a node in a relational algebra query tree.
    - operator: e.g., π (projection), σ (selection), ⨝ (join), Table
    - value: details for this operator (e.g., condition, column names, table name)
    - children: other nodes this one depends on
    """
    def __init__(self, operator, value=None):
        self.operator = operator  # Store type of operation
        self.value = value        # Store parameters or condition
        self.children = []        # Start with no children

    def add_child(self, child):
        """Attach another node as a child of this node."""
        self.children.append(child)

# ========================================================
# 2. Printing Query Trees
# ========================================================
def print_tree(node, level=0):
    """
    Prints a tree of relational algebra nodes with indentation.
    - node: starting node (root)
    - level: depth for indentation
    """
    indent = "  " * level  # 2 spaces per tree level
    if node.value:         # If node has extra info (like condition or cols)
        print(f"{indent}{node.operator} [{node.value}]")
    else:
        print(f"{indent}{node.operator}")
    for child in node.children:  # Recursively print all children
        print_tree(child, level + 1)

# ========================================================
# 3. SQL Parser
# ========================================================
def parse_sql(query):
    """
    Parses a simple SQL string into components:
    - SELECT part
    - FROM part
    - WHERE part (optional)
    """
    query = query.strip().rstrip(";")  # Remove spaces and final semicolon

    # Extract SELECT columns
    select_match = re.search(r"SELECT\s+(.*?)\s+FROM", query, re.IGNORECASE)
    select_cols = select_match.group(1).strip()

    # Extract FROM tables
    from_match = re.search(r"FROM\s+(.*?)(\s+WHERE|\s*$)", query, re.IGNORECASE)
    from_part = from_match.group(1).strip()
    tables = [t.strip() for t in from_part.split(",")]  # Handle multiple tables

    # Extract WHERE condition if exists
    where_match = re.search(r"WHERE\s+(.+)$", query, re.IGNORECASE)
    where_condition = where_match.group(1).strip() if where_match else None

    return select_cols, tables, where_condition

# ========================================================
# 4. Build Logical Query Tree
# ========================================================
def build_query_tree(select_cols, tables, where_condition):
    """
    Creates the logical query plan as a relational algebra tree.
    Steps:
      1. Create table nodes.
      2. If >1 table → join node.
      3. Add σ (selection) for WHERE.
      4. Add π (projection) for SELECT.
    """
    # Step 1: Create base table nodes
    table_nodes = [Node("Table", t) for t in tables]

    # Step 2: Handle joins if multiple tables
    if len(table_nodes) > 1:
        join_node = Node("⨝")
        for t in table_nodes:
            join_node.add_child(t)
        current_node = join_node
    else:
        current_node = table_nodes[0]

    # Step 3: Add selection if WHERE condition present
    if where_condition:
        selection_node = Node("σ", where_condition)
        selection_node.add_child(current_node)
        current_node = selection_node

    # Step 4: Add projection if SELECT not '*'
    if select_cols != "*":
        projection_node = Node("π", select_cols)
        projection_node.add_child(current_node)
        current_node = projection_node

    return current_node

# ========================================================
# 5. Algebraic Equivalence Optimizations
# ========================================================
def push_selection_down(node):
    """
    Push σ (selection) closer to base tables.
    """
    if node.operator == "σ" and len(node.children) == 1:
        child = node.children[0]
        if child.operator == "⨝":  # Push into join
            left, right = child.children
            cond = node.value
            left_sel = Node("σ", cond)
            left_sel.add_child(left)
            child.children[0] = left_sel
            return child
    for i, c in enumerate(node.children):
        node.children[i] = push_selection_down(c)
    return node

def push_projection_down(node):
    """
    Push π (projection) closer to base tables.
    """
    if node.operator == "π" and len(node.children) == 1:
        child = node.children[0]
        if child.operator in ["σ", "⨝"]:
            proj_child = Node("π", node.value)
            proj_child.add_child(child.children[0])
            child.children[0] = proj_child
    for i, c in enumerate(node.children):
        node.children[i] = push_projection_down(c)
    return node

# ========================================================
# 6. Join Order Heuristics
# ========================================================
def generate_join_orders(tables):
    """
    Generate all possible join sequences.
    """
    return list(permutations(tables))

# ========================================================
# 7. Volcano Iterator Model
# ========================================================
class Iterator:
    """Base interface for all iterator operators."""
    def open(self):
        pass
    def next(self):
        pass
    def close(self):
        pass

class TableScan(Iterator):
    """Reads all rows from a table (simulated list of dicts)."""
    def __init__(self, table_name, data):
        self.table_name = table_name
        self.data = data
        self.index = 0
    def open(self):
        self.index = 0
    def next(self):
        if self.index < len(self.data):
            row = self.data[self.index]
            self.index += 1
            return row
        return None
    def close(self):
        self.index = 0

class Selection(Iterator):
    """Filters rows using a condition."""
    def __init__(self, child, predicate):
        self.child = child
        self.predicate = predicate
    def open(self):
        self.child.open()
    def next(self):
        while True:
            row = self.child.next()
            if row is None:
                return None
            if eval(self.predicate, {}, {"row": row}):
                return row
    def close(self):
        self.child.close()

class Projection(Iterator):
    """Keeps only certain columns."""
    def __init__(self, child, columns):
        self.child = child
        self.columns = [c.strip() for c in columns.split(",")]
    def open(self):
        self.child.open()
    def next(self):
        row = self.child.next()
        if row is None:
            return None
        return {col: row[col] for col in self.columns}
    def close(self):
        self.child.close()

# ========================================================
# 8. Demonstration
# ========================================================
if __name__ == "__main__":
    # Example SQL
    sql_query = "SELECT name, age FROM Students, Departments WHERE Students.dept_id = Departments.id AND age > 20;"

    # Parse SQL
    select_cols, tables, where_condition = parse_sql(sql_query)

    # Build original tree
    print("\n=== Original Query Tree ===")
    root = build_query_tree(select_cols, tables, where_condition)
    print_tree(root)

    # Optimize: selection pushdown
    print("\n=== After Selection Pushdown ===")
    root = push_selection_down(root)
    print_tree(root)

    # Optimize: projection pushdown
    print("\n=== After Projection Pushdown ===")
    root = push_projection_down(root)
    print_tree(root)

    # Show join orders
    print("\n=== Possible Join Orders ===")
    for order in generate_join_orders(tables):
        print(order)

    # Volcano iterator demo
    print("\n=== Volcano Iterator Demo ===")
    students_data = [
        {"name": "Alice", "age": 22, "dept_id": 1},
        {"name": "Bob", "age": 19, "dept_id": 2}
    ]
    scan = TableScan("Students", students_data)
    sel = Selection(scan, "row['age'] > 20")
    proj = Projection(sel, "name, age")

    proj.open()
    while True:
        tuple_out = proj.next()
        if tuple_out is None:
            break
        print(tuple_out)
    proj.close()



=== Original Query Tree ===
π [name, age]
  σ [Students.dept_id = Departments.id AND age > 20]
    ⨝
      Table [Students]
      Table [Departments]

=== After Selection Pushdown ===
π [name, age]
  ⨝
    σ [Students.dept_id = Departments.id AND age > 20]
      Table [Students]
    Table [Departments]

=== After Projection Pushdown ===
π [name, age]
  ⨝
    π [name, age]
      σ [Students.dept_id = Departments.id AND age > 20]
        π [name, age]
          Table [Students]
    Table [Departments]

=== Possible Join Orders ===
('Students', 'Departments')
('Departments', 'Students')

=== Volcano Iterator Demo ===
{'name': 'Alice', 'age': 22}


# Query Optimization Transformations

## 1. Original Query Tree
$$
\pi_{\text{name}, \text{age}} \
  \sigma_{\text{Students.dept\_id} = \text{Departments.id} \ \wedge\ \text{age} > 20} \
    (\text{Students} \ \bowtie\ \text{Departments})
$$
- $\pi$ (projection) for **name**, **age**  
- $\sigma$ (selection) for the **WHERE** condition  
- $\bowtie$ (join) connecting the two base tables.

## 2. After Selection Pushdown
$$
\pi_{\text{name}, \text{age}} \
  (\sigma_{\text{Students.dept\_id} = \text{Departments.id} \ \wedge\ \text{age} > 20} (\text{Students}) \ \bowtie\ \text{Departments})
$$
The selection moved below the join so it filters **Students** before joining to **Departments**.  
**Optimization goal:** Reduce intermediate result sizes.

## 3. After Projection Pushdown
$$
\pi_{\text{name}, \text{age}} \
  \left(
    \pi_{\text{name}, \text{age}} \ \sigma_{\text{Students.dept\_id} = \text{Departments.id} \ \wedge\ \text{age} > 20} \ \pi_{\text{name}, \text{age}}(\text{Students}) \ \bowtie\ \text{Departments}
  \right)
$$
Projections are pushed closer to the base tables.  
Multiple $\pi$ operators show narrowing columns early to reduce tuple width.

## 4. Possible Join Orders
- $(\text{Students}, \text{Departments})$  
- $(\text{Departments}, \text{Students})$  
These represent permutations of table join order — the search space a join optimizer could explore.

## 5. Volcano Iterator Demo
Example pipeline: **TableScan → Selection → Projection**  
Output:  
Only **Alice** passed the $\text{age} > 20$ filter.
